[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/22_conv2d_solution.ipynb)

# 🟡 Solution: 2D Convolution

*Core Ops & Layers · Medium*

Reference implementation. Try it yourself in `22_conv2d.ipynb` first.

---
Implement a 2-D convolution (strictly, a cross-correlation) from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    ...
```

- `x`: `(B, C_in, H, W)` — **NCHW**, the PyTorch layout
- `weight`: `(C_out, C_in, kH, kW)` — **OIHW**
- `padding`: an **int**, applied symmetrically to all four sides
- output: `(B, C_out, H_out, W_out)` with
  $H_{out} = \lfloor (H + 2p - k_H)/s \rfloor + 1$

### Rules
- Do **not** use `jax.lax.conv_general_dilated` or `jax.scipy.signal`
- No Python loop over output pixels
- `bias` is `(C_out,)` and is added per output channel

### The shape that makes this easy
The trick is to materialise every sliding window at once, so the convolution
becomes a single tensor contraction:

```
patches: (B, C_in, H_out, W_out, kH, kW)
weight:  (C_out, C_in, kH, kW)
        -> einsum('bihwjk,oijk->bohw')
```

Getting that einsum right first try is the actual test. Read it as: for each
batch `b` and output position `(h, w)`, contract over the input channel `i` and
the kernel offsets `(j, k)` to produce output channel `o`.

Build the windows with **index arrays**, not a loop:
`rows = arange(H_out)[:, None] * stride + arange(kH)[None, :]` gives an
`(H_out, kH)` matrix that gathers every row window in one go.

### Layout note
This task follows the PyTorch original's NCHW/OIHW convention so the two
implementations line up line for line. Flax goes the other way — `nnx.Conv`
uses **NHWC** inputs and **HWIO** kernels, because that layout is what XLA
prefers on TPU. Converting is a transpose in each direction; the arithmetic
here is identical either way.

### Why im2col at all
Materialising the patches costs $k_H k_W$ memory blow-up, which sounds
terrible — but it turns the convolution into one big matmul, and matmuls are
what hardware is built to do. Real implementations either do exactly this
(im2col + GEMM) or fuse the gather into the matmul; almost nobody runs the
naive seven-deep loop.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    if padding > 0:
        # Pad only the spatial axes, symmetrically.
        x = jnp.pad(x, ((0, 0), (0, 0), (padding, padding), (padding, padding)))

    B, C_in, H, W = x.shape
    C_out, _, kH, kW = weight.shape
    H_out = (H - kH) // stride + 1
    W_out = (W - kW) // stride + 1

    # Index matrices that name every window position at once — this is the
    # loop-free replacement for torch's x.unfold(2, kH, stride).
    rows = jnp.arange(H_out)[:, None] * stride + jnp.arange(kH)[None, :]  # (H_out, kH)
    cols = jnp.arange(W_out)[:, None] * stride + jnp.arange(kW)[None, :]  # (W_out, kW)

    # (B, C_in, H_out, kH, W) -> (B, C_in, H_out, kH, W_out, kW)
    patches = x[:, :, rows][..., cols]
    # -> (B, C_in, H_out, W_out, kH, kW), so the einsum below reads exactly
    #    like the PyTorch original's.
    patches = patches.transpose(0, 1, 2, 4, 3, 5)

    out = jnp.einsum("bihwjk,oijk->bohw", patches, weight)

    if bias is not None:
        out = out + bias.reshape(1, -1, 1, 1)
    return out

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

x = jax.random.normal(jax.random.key(0), (2, 3, 8, 8))     # NCHW
w = jax.random.normal(jax.random.key(1), (4, 3, 3, 3))     # OIHW

print("valid, stride 1:", my_conv2d(x, w).shape,                    "(2, 4, 6, 6)")
print("same-ish, pad 1:", my_conv2d(x, w, padding=1).shape,         "(2, 4, 8, 8)")
print("stride 2:       ", my_conv2d(x, w, stride=2).shape,          "(2, 4, 3, 3)")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("conv2d")